# Task 6: Retrieval-Augmented Generation (RAG) Pipeline

## Objective

Implement a simple Retrieval-Augmented Generation (RAG) pipeline.

The pipeline demonstrates:

- Document creation
- Text chunking
- Text embeddings
- Similarity calculation
- Document retrieval
- Context generation
- Question answering

RAG improves language-model responses by retrieving relevant information from an external knowledge source before generating an answer.

## Step 1: Import Libraries

We use Sentence Transformers to convert text into numerical embeddings.

Cosine similarity is used to find documents that are most relevant to a query.

In [3]:
import numpy as np
from sentence_transformers import SentenceTransformer

print("Libraries imported successfully!")

Libraries imported successfully!


## Step 2: Create a Knowledge Base

A RAG system needs an external knowledge source.

Here we create a small collection of documents related to Generative AI.

In [4]:
documents = [
    "Generative AI creates new content such as text, images, audio and code.",
    
    "Retrieval-Augmented Generation combines information retrieval with language generation.",
    
    "Embeddings represent text as numerical vectors that capture semantic meaning.",
    
    "Vector databases store embeddings and allow efficient similarity search.",
    
    "Large Language Models are neural networks trained on large amounts of text data.",
    
    "Prompt engineering involves designing effective instructions for language models.",
    
    "Fine-tuning adapts a pretrained model to perform better on a specific task.",
    
    "RAG can reduce hallucinations by providing relevant external information to a language model."
]

print("Number of documents:", len(documents))

Number of documents: 8


## Step 3: Generate Text Embeddings

An embedding model converts each document into a numerical vector.

Documents with similar meanings should have similar vector representations.

In [5]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

document_embeddings = embedding_model.encode(
    documents,
    convert_to_numpy=True
)

print(
    "Embedding shape:",
    document_embeddings.shape
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\nanda\OneDrive\Documents\Nandu\AU\Generative-AI-Skill-Development\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\nanda\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding shape: (8, 384)


## Step 4: Create a User Query

The user asks a question.

The RAG system will search the knowledge base for information relevant to this question.

In [6]:
query = "What is Retrieval-Augmented Generation?"

print("Query:")
print(query)

Query:
What is Retrieval-Augmented Generation?


## Step 5: Convert Query into an Embedding

The same embedding model is used to convert the user query into a vector.

In [7]:
query_embedding = embedding_model.encode(
    query,
    convert_to_numpy=True
)

print(
    "Query embedding shape:",
    query_embedding.shape
)

Query embedding shape: (384,)


## Step 6: Calculate Cosine Similarity

Cosine similarity measures how similar two vectors are.

A higher value means the texts are more semantically similar.

In [8]:
def cosine_similarity(a, b):

    return np.dot(a, b) / (
        np.linalg.norm(a)
        * np.linalg.norm(b)
    )

## Step 7: Retrieve Relevant Documents

We calculate the similarity between the query and every document.

The documents with the highest similarity scores are retrieved.

In [9]:
similarities = []

for i, embedding in enumerate(
    document_embeddings
):

    score = cosine_similarity(
        query_embedding,
        embedding
    )

    similarities.append(
        (i, score)
    )

similarities.sort(
    key=lambda x: x[1],
    reverse=True
)

top_k = 3

retrieved_documents = similarities[
    :top_k
]

print("Retrieved Documents:\n")

for index, score in retrieved_documents:

    print(
        f"Score: {score:.4f}"
    )

    print(
        documents[index]
    )

    print()

Retrieved Documents:

Score: 0.7039
Retrieval-Augmented Generation combines information retrieval with language generation.

Score: 0.3643
Fine-tuning adapts a pretrained model to perform better on a specific task.

Score: 0.3453
Generative AI creates new content such as text, images, audio and code.



## Step 8: Build Retrieved Context

The retrieved documents are combined into a context.

A language model can use this context to answer the user's question.

In [10]:
context = "\n".join(
    documents[index]
    for index, score in retrieved_documents
)

print("Retrieved Context:\n")
print(context)

Retrieved Context:

Retrieval-Augmented Generation combines information retrieval with language generation.
Fine-tuning adapts a pretrained model to perform better on a specific task.
Generative AI creates new content such as text, images, audio and code.


## Step 9: Generate an Answer

For this simplified implementation, the answer is generated from the retrieved context using keyword-based sentence selection.

This demonstrates the RAG concept without requiring a large language model or Ollama.

In [11]:
def generate_answer(query, context):

    query_words = set(
        query.lower().split()
    )

    sentences = context.split(".")

    scored_sentences = []

    for sentence in sentences:

        words = set(
            sentence.lower().split()
        )

        score = len(
            query_words & words
        )

        if score > 0:

            scored_sentences.append(
                (sentence.strip(), score)
            )

    scored_sentences.sort(
        key=lambda x: x[1],
        reverse=True
    )

    if scored_sentences:

        answer = ". ".join(
            sentence
            for sentence, score
            in scored_sentences[:2]
        )

        return answer + "."

    return "No relevant information was found."


answer = generate_answer(
    query,
    context
)

print("Question:")
print(query)

print("\nAnswer:")
print(answer)

Question:
What is Retrieval-Augmented Generation?

Answer:
Retrieval-Augmented Generation combines information retrieval with language generation.


## Step 10: Test Another Query

We test the RAG pipeline with another question to verify that relevant documents are retrieved.

In [12]:
query2 = "How do embeddings represent text?"

query2_embedding = embedding_model.encode(
    query2,
    convert_to_numpy=True
)

scores = []

for i, embedding in enumerate(
    document_embeddings
):

    score = cosine_similarity(
        query2_embedding,
        embedding
    )

    scores.append(
        (i, score)
    )

scores.sort(
    key=lambda x: x[1],
    reverse=True
)

print("Question:")
print(query2)

print("\nTop Retrieved Documents:\n")

for index, score in scores[:3]:

    print(
        f"Similarity: {score:.4f}"
    )

    print(
        documents[index]
    )

    print()

Question:
How do embeddings represent text?

Top Retrieved Documents:

Similarity: 0.7939
Embeddings represent text as numerical vectors that capture semantic meaning.

Similarity: 0.4560
Vector databases store embeddings and allow efficient similarity search.

Similarity: 0.3778
Large Language Models are neural networks trained on large amounts of text data.



## Step 11: Create the Complete RAG Pipeline

The complete process is combined into a single function:

1. Convert query into embedding
2. Retrieve similar documents
3. Build context
4. Generate an answer

In [13]:
def rag_pipeline(
    query,
    documents,
    document_embeddings,
    top_k=3
):

    # Query embedding
    query_embedding = embedding_model.encode(
        query,
        convert_to_numpy=True
    )

    # Calculate similarities
    scores = []

    for i, embedding in enumerate(
        document_embeddings
    ):

        score = cosine_similarity(
            query_embedding,
            embedding
        )

        scores.append(
            (i, score)
        )

    # Rank documents
    scores.sort(
        key=lambda x: x[1],
        reverse=True
    )

    # Retrieve top documents
    retrieved = scores[:top_k]

    # Build context
    context = "\n".join(
        documents[index]
        for index, score in retrieved
    )

    # Generate answer
    answer = generate_answer(
        query,
        context
    )

    return answer, retrieved

## Step 12: Run the Complete RAG Pipeline

In [14]:
question = "How does RAG reduce hallucinations?"

answer, retrieved = rag_pipeline(
    question,
    documents,
    document_embeddings,
    top_k=3
)

print("Question:")
print(question)

print("\nRetrieved Documents:")

for index, score in retrieved:

    print(
        f"\nSimilarity: {score:.4f}"
    )

    print(
        documents[index]
    )

print("\nGenerated Answer:")
print(answer)

Question:
How does RAG reduce hallucinations?

Retrieved Documents:

Similarity: 0.7986
RAG can reduce hallucinations by providing relevant external information to a language model.

Similarity: 0.0528
Generative AI creates new content such as text, images, audio and code.

Similarity: 0.0204
Retrieval-Augmented Generation combines information retrieval with language generation.

Generated Answer:
RAG can reduce hallucinations by providing relevant external information to a language model.


## Step 13: Retrieval Scores

The similarity scores show how relevant each retrieved document is to the user's question.

In [15]:
print("Retrieval Results:\n")

for rank, (index, score) in enumerate(
    retrieved,
    start=1
):

    print(
        f"Rank {rank}"
    )

    print(
        f"Similarity: {score:.4f}"
    )

    print(
        f"Document: {documents[index]}"
    )

    print()

Retrieval Results:

Rank 1
Similarity: 0.7986
Document: RAG can reduce hallucinations by providing relevant external information to a language model.

Rank 2
Similarity: 0.0528
Document: Generative AI creates new content such as text, images, audio and code.

Rank 3
Similarity: 0.0204
Document: Retrieval-Augmented Generation combines information retrieval with language generation.



## Step 14: Results

The RAG pipeline successfully demonstrates:

- Knowledge base creation
- Text embeddings
- Query embeddings
- Semantic similarity
- Relevant document retrieval
- Context construction
- Context-based answer generation

In [16]:
print("========== RAG RESULTS ==========")

print(
    "Number of documents:",
    len(documents)
)

print(
    "Embedding dimension:",
    document_embeddings.shape[1]
)

print(
    "Top-K documents retrieved:",
    top_k
)

print(
    "Question:",
    question
)

print(
    "\nAnswer:",
    answer
)

print("=================================")

========== RAG RESULTS ==========
Number of documents: 8
Embedding dimension: 384
Top-K documents retrieved: 3
Question: How does RAG reduce hallucinations?

Answer: RAG can reduce hallucinations by providing relevant external information to a language model.


# Conclusion

A simplified Retrieval-Augmented Generation pipeline was implemented.

The system converts documents and user queries into embeddings and uses cosine similarity to retrieve the most relevant information.

The retrieved information is then provided as context for generating an answer.

This demonstrates the fundamental RAG workflow:

User Query → Embedding → Retrieval → Context → Answer

RAG is useful because it allows language models to access external knowledge instead of relying only on information stored in their parameters.